# YOLO and CNN Split Consistency Check

This notebook checks whether the original YOLO dataset and the CNN classification dataset use the same train-validation-test split. The goal is to identify possible differences between the datasets before comparing model results, because a fair comparison requires that both approaches are evaluated on the same data split.

In [1]:
from pathlib import Path

# Project root = folder where the notebook/script is executed
PROJECT_ROOT = Path.cwd()

# YOLO dataset
yolo_base = PROJECT_ROOT / "yolo_dataset" / "images"

# CNN / classification dataset
cnn_base = PROJECT_ROOT / "dataset" / "clean_split_dataset_augmented"

image_extensions = [".jpg", ".jpeg", ".png"]

def get_yolo_files(split):
    return {
        p.name
        for p in (yolo_base / split).iterdir()
        if p.suffix.lower() in image_extensions
    }

def get_cnn_files(split):
    files = set()
    split_dir = cnn_base / split

    for class_dir in split_dir.iterdir():
        if class_dir.is_dir():
            for p in class_dir.iterdir():
                if p.suffix.lower() in image_extensions:
                    files.add(p.name)

    return files

for split in ["train", "val", "test"]:
    yolo_files = get_yolo_files(split)
    cnn_files = get_cnn_files(split)

    common_files = yolo_files & cnn_files
    only_yolo = yolo_files - cnn_files
    only_cnn = cnn_files - yolo_files

    print(f"\n===== {split.upper()} =====")
    print("YOLO files:", len(yolo_files))
    print("CNN files :", len(cnn_files))
    print("Common   :", len(common_files))
    print("Only YOLO:", len(only_yolo))
    print("Only CNN :", len(only_cnn))


===== TRAIN =====
YOLO files: 1761
CNN files : 2490
Common   : 0
Only YOLO: 1761
Only CNN : 2490

===== VAL =====
YOLO files: 499
CNN files : 502
Common   : 0
Only YOLO: 499
Only CNN : 502

===== TEST =====
YOLO files: 264
CNN files : 258
Common   : 0
Only YOLO: 264
Only CNN : 258


In [2]:
from pathlib import Path
import re

yolo_base = Path(r"C:\Users\gokce\Desktop\FHNWfs26\CLX\yolo_dataset\images")
cnn_base = Path(r"C:\Users\gokce\Desktop\FHNWfs26\CLX\dataset\clean_split_dataset_augmented")

image_extensions = [".jpg", ".jpeg", ".png"]

def normalize_name(filename):
    name = Path(filename).stem.lower()

    # Remove Roboflow suffix after .rf.
    name = name.split(".rf.")[0]

    # Convert _jpg, _png, _jpeg endings back to normal base name
    name = re.sub(r"_(jpg|jpeg|png)$", "", name)

    # Remove augmented prefix
    name = re.sub(r"^aug_", "", name)

    return name

def get_yolo_normalized_files(split):
    return {
        normalize_name(p.name)
        for p in (yolo_base / split).iterdir()
        if p.suffix.lower() in image_extensions
    }

def get_cnn_normalized_files(split):
    files = set()
    split_dir = cnn_base / split

    for class_dir in split_dir.iterdir():
        if class_dir.is_dir():
            for p in class_dir.iterdir():
                if p.suffix.lower() in image_extensions:
                    # Optional: ignore augmented files
                    if p.name.startswith("aug_"):
                        continue
                    files.add(normalize_name(p.name))

    return files

for split in ["train", "val", "test"]:
    yolo_files = get_yolo_normalized_files(split)
    cnn_files = get_cnn_normalized_files(split)

    common_files = yolo_files & cnn_files
    only_yolo = yolo_files - cnn_files
    only_cnn = cnn_files - yolo_files

    print(f"\n===== {split.upper()} NORMALIZED ORIGINAL ONLY =====")
    print("YOLO files:", len(yolo_files))
    print("CNN files :", len(cnn_files))
    print("Common   :", len(common_files))
    print("Only YOLO:", len(only_yolo))
    print("Only CNN :", len(only_cnn))

    print("Example common:", list(common_files)[:5])
    print("Example only YOLO:", list(only_yolo)[:5])
    print("Example only CNN:", list(only_cnn)[:5])


===== TRAIN NORMALIZED ORIGINAL ONLY =====
YOLO files: 1761
CNN files : 1764
Common   : 1227
Only YOLO: 534
Only CNN : 537
Example common: ['metal28', 'glass229', 'glass230', 'paper462', 'metal21']
Example only YOLO: ['glass143', 'plastic428', 'metal403', 'trash88', 'metal7']
Example only CNN: ['plastic256', 'glass462', 'plastic237', 'trash64', 'cardboard296']

===== VAL NORMALIZED ORIGINAL ONLY =====
YOLO files: 499
CNN files : 502
Common   : 106
Only YOLO: 393
Only CNN : 396
Example common: ['metal129', 'paper266', 'paper333', 'glass145', 'trash63']
Example only YOLO: ['plastic237', 'glass462', 'plastic256', 'plastic293', 'cardboard29']
Example only CNN: ['glass143', 'plastic428', 'metal403', 'glass27', 'paper173']

===== TEST NORMALIZED ORIGINAL ONLY =====
YOLO files: 264
CNN files : 258
Common   : 25
Only YOLO: 239
Only CNN : 233
Example common: ['trash33', 'paper486', 'trash119', 'plastic228', 'glass172']
Example only YOLO: ['metal324', 'paper167', 'cardboard247', 'trash64', 'car